# LAB-13 · First Triton kernel

Think of each Triton program as a worker unpacking one crate of numbers. You will add two vectors, then map rows and paired columns into memory addresses, including tensors whose neighbors are not stored together.

~20 minutes · **GPU optional**. A kernel is a small function run over many data items; Triton is a Python-based language for writing it. Run cells in order, fill the `# TODO` gaps, then use the checks; starter failures are expected.

Use a fresh Python runtime. Setup can install Triton in your account; it needs no API key or repository files. CPU correctness uses Triton's interpreter on a supported Linux runtime (Colab/Kaggle); native macOS is not a supported Triton runtime.


In [ ]:
import os
import sys
import torch

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
# The interpreter setting must be chosen BEFORE the first import of Triton.
if "triton" in sys.modules:
    raise RuntimeError("Restart the runtime, then run this setup cell first (Triton is already imported).")
if HAS_GPU:
    os.environ.pop("TRITON_INTERPRET", None)
else:
    os.environ["TRITON_INTERPRET"] = "1"
    print("No GPU: correctness checks only, timings skipped")
try:
    import triton
except ImportError:
    # Runs only in your notebook runtime; PyTorch is preinstalled in Colab/Kaggle.
    %pip install -q triton
    import triton
import triton.language as tl

GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "none (CPU interpreter)"
print(f"torch {torch.__version__}; triton {triton.__version__}; GPU: {GPU_NAME}")
torch.manual_seed(0)


In [ ]:
# Self-check failures are reported without stopping Run all, so you can reach Solution.
# Solution checks are strict: a broken reference stops execution before benchmarking.
def check(label, test, strict=False):
    try:
        test()
    except Exception as error:
        print(f"FAIL {label}: {type(error).__name__}: {error}")
        if strict:
            raise
        return False
    print(f"PASS {label}")
    return True

# Vendor datasheet peaks, decimal GB/s. Match variants conservatively.
PEAK_GB_S = {"T4": 320.0, "P100 16 GB": 732.0, "H100 SXM": 3350.0}
def detected_peak():
    if not HAS_GPU:
        return None
    name = GPU_NAME.upper()
    if name in ("TESLA T4", "NVIDIA T4", "T4"):
        return PEAK_GB_S["T4"]
    memory = torch.cuda.get_device_properties(0).total_memory
    if "P100" in name and 15e9 <= memory <= 18e9:
        return PEAK_GB_S["P100 16 GB"]
    if "H100" in name and "SXM" in name:
        return PEAK_GB_S["H100 SXM"]
    return None

def report(label, milliseconds, byte_count):
    gb_s = byte_count / (milliseconds * 1e6)
    text = f"your run | {label}: {milliseconds:.4f} ms, {byte_count:,} logical bytes, {gb_s:.2f} GB/s"
    peak = detected_peak()
    if peak is not None:
        text += f", {100 * gb_s / peak:.1f}% of vendor peak"
    print(text)

def bench(label, call, byte_count):
    if not HAS_GPU:
        print(f"Timings skipped: {label} (CPU correctness only)")
        return
    from triton.testing import do_bench
    milliseconds = do_bench(call, warmup=25, rep=100)
    report(label, milliseconds, byte_count)


## 1 · One crate per program

A vector is a one-dimensional list; `program_id` chooses a crate and `arange` numbers its slots. The mask skips slots beyond the list's end, like empty seats in the final row of a theater. Fix the marked operation; the launch covers ceil(N / BLOCK) crates (1,000 values with BLOCK 256 need four programs).


In [ ]:
@triton.jit
def add_kernel(A, B, Out, N: tl.constexpr, BLOCK: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < N
    a = tl.load(A + offsets, mask=mask, other=0)
    b = tl.load(B + offsets, mask=mask, other=0)
    value = a - b  # TODO: add the two loaded vectors
    tl.store(Out + offsets, value, mask=mask)

def vector_add(a, b, out=None):
    assert a.ndim == b.ndim == 1 and a.shape == b.shape
    assert a.is_contiguous() and b.is_contiguous()
    if out is None:
        out = torch.empty_like(a)
    if a.numel():
        add_kernel[(triton.cdiv(a.numel(), 256),)](a, b, out, a.numel(), 256)
    return out


In [ ]:
def check_add():
    # Illustrative sizes: empty, singleton, partial block and multiple programs.
    for n in (0, 1, 255, 256, 1000):
        a = torch.randn(n, device=DEVICE, dtype=torch.float32)
        b = torch.randn_like(a)
        torch.testing.assert_close(vector_add(a, b), a + b, rtol=1e-5, atol=1e-6)
check("vector add, including a masked tail", check_add)


## 2 · Rows, columns and strides

A stride is the distance in stored elements between neighbors along one axis. Like two theaters with different row widths, the input and output need their own address formulas. Add corresponding values from the two halves of each row; fix both marked addresses. This is the paired-coordinate indexing used in a RoPE kernel (rotary position encoding), without implementing the rotation.


In [ ]:
@triton.jit
def pair_kernel(X, Out, ROWS: tl.constexpr, COLS: tl.constexpr,
                S0: tl.constexpr, S1: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    col = tl.arange(0, BLOCK)
    mask = col < COLS
    # X has 2 * COLS columns: pair the first half with the second half.
    first = row * S0 + 0 * col  # TODO: use the column stride
    second = first  # TODO: move to the same column in the second half
    x = tl.load(X + first, mask=mask, other=0)
    y = tl.load(X + second, mask=mask, other=0)
    tl.store(Out + row * COLS + col, x + y, mask=mask)

def pair_halves(x, out=None):
    assert x.ndim == 2 and x.shape[1] > 0 and x.shape[1] % 2 == 0
    rows, width = x.shape
    cols = width // 2
    if out is None:
        out = torch.empty((rows, cols), device=x.device, dtype=x.dtype)
    if rows:
        pair_kernel[(rows,)](x, out, rows, cols, x.stride(0), x.stride(1), triton.next_power_of_2(cols))
    return out


In [ ]:
def check_pairs():
    # Contiguous, transposed, and sliced: both row and column strides can change.
    for rows, width in ((1, 2), (3, 14), (5, 18)):
        base = torch.arange(rows * width, device=DEVICE, dtype=torch.float32).reshape(rows, width)
        transposed = base.T.contiguous().T
        storage = torch.arange(rows * width * 2, device=DEVICE, dtype=torch.float32).reshape(rows, width * 2)
        for x in (base, transposed, storage[:, ::2]):
            half = width // 2
            torch.testing.assert_close(pair_halves(x), x[:, :half] + x[:, half:], rtol=1e-5, atol=1e-6)
check("2D pairs with real tensor strides", check_pairs)


## Try first

Try filling the TODOs and rerunning each exercise and its check before reading on. Run all intentionally reports starter failures, then installs the complete definitions below and checks them again. Only checked solutions are timed.


## Solution


In [ ]:
@triton.jit
def add_kernel(A, B, Out, N: tl.constexpr, BLOCK: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < N
    a = tl.load(A + offsets, mask=mask, other=0)
    b = tl.load(B + offsets, mask=mask, other=0)
    value = a + b
    tl.store(Out + offsets, value, mask=mask)

def vector_add(a, b, out=None):
    assert a.ndim == b.ndim == 1 and a.shape == b.shape
    assert a.is_contiguous() and b.is_contiguous()
    if out is None:
        out = torch.empty_like(a)
    if a.numel():
        add_kernel[(triton.cdiv(a.numel(), 256),)](a, b, out, a.numel(), 256)
    return out


In [ ]:
@triton.jit
def pair_kernel(X, Out, ROWS: tl.constexpr, COLS: tl.constexpr,
                S0: tl.constexpr, S1: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    col = tl.arange(0, BLOCK)
    mask = col < COLS
    # X has 2 * COLS columns: pair the first half with the second half.
    first = row * S0 + col * S1
    second = first + COLS * S1
    x = tl.load(X + first, mask=mask, other=0)
    y = tl.load(X + second, mask=mask, other=0)
    tl.store(Out + row * COLS + col, x + y, mask=mask)

def pair_halves(x, out=None):
    assert x.ndim == 2 and x.shape[1] > 0 and x.shape[1] % 2 == 0
    rows, width = x.shape
    cols = width // 2
    if out is None:
        out = torch.empty((rows, cols), device=x.device, dtype=x.dtype)
    if rows:
        pair_kernel[(rows,)](x, out, rows, cols, x.stride(0), x.stride(1), triton.next_power_of_2(cols))
    return out


In [ ]:
check("solution: vector add", check_add, strict=True)
check("solution: 2D strided pairs", check_pairs, strict=True)


## Measure on a GPU

Results below are **your run**, not a promised speedup. Byte counts are logical tensor reads/writes, excluding allocation, cache effects and instruction overhead; GB/s is effective bandwidth, not measured HBM traffic, and can exceed a datasheet peak when data is cached. Times depend on shape, GPU, compiler and runtime.

The small peak table uses vendor datasheets: T4 320 GB/s, P100 **16 GB** 732 GB/s, H100 **SXM** 3.35 TB/s. Ambiguous or unknown GPU variants show GB/s without a peak percentage. No timings run in CPU-interpreter mode.


In [ ]:
if HAS_GPU:
    # Illustrative workload; preallocate outputs so allocation is outside the timed callable.
    n = 2**22
    a = torch.randn(n, device=DEVICE)
    b = torch.randn_like(a)
    out = torch.empty_like(a)
    bench("vector add", lambda: vector_add(a, b, out), 3 * n * a.element_size())
    x = torch.randn((2048, 2048), device=DEVICE)
    paired = torch.empty((2048, 1024), device=DEVICE)
    bench("2D pairs", lambda: pair_halves(x, paired), 3 * paired.numel() * x.element_size())
else:
    print("Timings skipped (CPU correctness only)")


## Take it further

You built masked vector addition and stride-aware paired indexing. Real systems also choose layouts and launch sizes for their exact tensors; this exercise establishes the address arithmetic first. Backs I29 (Triton programs) and I33 (coordinate mapping).

[Academy LAB-13 page](https://tile-and-error-academy.vercel.app/#lab/lab-13) · [Tile & Error channel](https://www.youtube.com/@tileanderror)
